In [12]:
import pygame
import math
import random

In [13]:
# Inicialização
pygame.init()

(5, 0)

In [14]:
# Configurações da tela
LARGURA, ALTURA = 800, 600
TELA = pygame.display.set_mode((LARGURA, ALTURA))
pygame.display.set_caption("Asteroids - Transformações Geométricas")

In [15]:
# Cores
PRETO = (0, 0, 0)
BRANCO = (255, 255, 255)
VERMELHO = (255, 60, 60)
AMARELO = (255, 220, 0)

In [16]:
# Relógio para controle de FPS
relogio = pygame.time.Clock()
FPS = 60

In [17]:
# =========================
# Transformações manuais
# =========================

def transformar_vertices(vertices, x, y, angulo, escala=1):
    """Aplica escala, rotação e translação manualmente nos vértices."""
    pontos_transformados = []
    rad = math.radians(angulo)
    cos_a = math.cos(rad)
    sin_a = math.sin(rad)
    for vx, vy in vertices:
        # Escala
        sx = vx * escala
        sy = vy * escala
        # Rotação + translação
        x_final = sx * cos_a - sy * sin_a + x
        y_final = sx * sin_a + sy * cos_a + y
        pontos_transformados.append((x_final, y_final))
    return pontos_transformados


def colisao_circular(x1, y1, r1, x2, y2, r2):
    """Detecção de colisão manual por distância euclidiana."""
    distancia = math.hypot(x1 - x2, y1 - y2)
    return distancia < r1 + r2


In [18]:
# =========================
# Classe Nave
# =========================

class Nave:
    def __init__(self):
        self.x = LARGURA // 2
        self.y = ALTURA // 2
        self.angulo = -90
        self.vel = 4
        self.raio = 20
        self.vertices_base = [
            (25, 0),
            (-15, -15),
            (-8, 0),
            (-15, 15)
        ]

    def mover(self, teclas):
        # Movimento horizontal
        if teclas[pygame.K_LEFT]:
            self.x -= self.vel

        if teclas[pygame.K_RIGHT]:
            self.x += self.vel
    # Movimento vertical
        if teclas[pygame.K_UP]:
            self.y -= self.vel
        if teclas[pygame.K_DOWN]:
            self.y += self.vel
    # Rotacionar para mirar
        if teclas[pygame.K_a]:
            self.angulo -= 5
        if teclas[pygame.K_d]:
            self.angulo += 5

        self.x = max(self.raio, min(LARGURA - self.raio, self.x))
        self.y = max(self.raio, min(ALTURA - self.raio, self.y))

    # Mantém nave dentro da tela
        self.x = max(self.raio, min(LARGURA - self.raio, self.x))
        self.y = max(self.raio, min(ALTURA - self.raio, self.y))
    def desenhar(self):
        pontos = transformar_vertices(
            self.vertices_base,
            self.x,
            self.y,
            self.angulo
        )
        pygame.draw.polygon(TELA, BRANCO, pontos, 2)

In [19]:
# =========================
# Classe Tiro
# =========================

class Tiro:
    def __init__(self, x, y, angulo):
        self.x = x
        self.y = y
        self.angulo = angulo
        self.vel = 10
        self.raio = 4

        rad = math.radians(angulo)
        self.vx = math.cos(rad) * self.vel
        self.vy = math.sin(rad) * self.vel

    def mover(self):
        self.x += self.vx
        self.y += self.vy

    def desenhar(self):
        pygame.draw.circle(
            TELA,
            AMARELO,
            (int(self.x), int(self.y)),
            self.raio
        )

    def fora_da_tela(self):
        return(
            self.x < 0 or self.x > LARGURA or
            self.y < 0 or self.y > ALTURA
        )

In [20]:
# --- Classe do Asteroide ---
class Asteroide:
    def __init__(self):
        lado = random.choice(["topo", "baixo", "esquerda", "direita"])

        if lado == "topo":
            self.x = random.randint(0, LARGURA)
            self.y = -40
        elif lado == "baixo":
            self.x = random.randint(0, LARGURA)
            self.y = ALTURA + 40
        elif lado == "esquerda":
            self.x = -40
            self.y = random.randint(0, ALTURA)
        else:
            self.x = LARGURA + 40
            self.y = random.randint(0, ALTURA)

        alvo_x = random.randint(100, LARGURA - 100)
        alvo_y = random.randint(100, ALTURA - 100)

        dx = alvo_x - self.x
        dy = alvo_y - self.y
        dist = math.hypot(dx, dy)

        self.vx = dx / dist * random.uniform(1.5, 3.0)
        self.vy = dy / dist * random.uniform(1.5, 3.0)

        self.raio = random.randint(20, 35)
        self.angulo = 0
        self.rotacao = random.uniform(-2, 2)

        self.vertices_base = [
            (0, -1),
            (0.8, -0.5),
            (1, 0.2),
            (0.4, 1),
            (-0.6, 0.8),
            (-1, 0),
            (-0.5, -0.8)
        ]

    def mover(self):
        self.x += self.vx
        self.y += self.vy
        self.angulo += self.rotacao

    def desenhar(self):
        pontos = transformar_vertices(
            self.vertices_base,
            self.x,
            self.y,
            self.angulo,
            self.raio
        )
        pygame.draw.polygon(TELA, BRANCO, pontos, 2)

    def fora_da_tela(self):
        return (
            self.x < -80 or self.x > LARGURA + 80 or
            self.y < -80 or self.y > ALTURA + 80
        )

In [21]:
# --- Função Principal ---
def main():
    nave = Nave()
    tiros = []
    asteroides = []
    pontos = 0
    vidas = 3
    game_over = False
    fonte = pygame.font.SysFont(None, 36)
    fonte_grande = pygame.font.SysFont(None, 72)
    SPAWN_ASTEROIDE = pygame.USEREVENT + 1
    pygame.time.set_timer(SPAWN_ASTEROIDE, 1200)
    rodando = True

    while rodando:
        relogio.tick(FPS)
        TELA.fill(PRETO)
        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False
            if evento.type == SPAWN_ASTEROIDE and not game_over:
                asteroides.append(Asteroide())
            if evento.type == pygame.KEYDOWN:
                if evento.key == pygame.K_SPACE and not game_over:
                    tiros.append(Tiro(nave.x, nave.y, nave.angulo))
                if evento.key == pygame.K_r and game_over:
                    nave = Nave()
                    tiros.clear()
                    asteroides.clear()
                    pontos = 0
                    vidas = 3
                    game_over = False
        if not game_over:
            teclas = pygame.key.get_pressed()
            nave.mover(teclas)
            for tiro in tiros[:]:
                tiro.mover()
                if tiro.fora_da_tela():
                    tiros.remove(tiro)
            for ast in asteroides[:]:
                ast.mover()
                if ast.fora_da_tela():
                    asteroides.remove(ast)
                    continue
                if colisao_circular(nave.x, nave.y, nave.raio, ast.x, ast.y, ast.raio):
                    asteroides.remove(ast)
                    vidas -= 1
                    if vidas <= 0:
                        game_over = True
            for tiro in tiros[:]:
                for ast in asteroides[:]:
                    if colisao_circular(tiro.x, tiro.y, tiro.raio, ast.x, ast.y, ast.raio):
                        if tiro in tiros:
                            tiros.remove(tiro)
                        if ast in asteroides:
                            asteroides.remove(ast)
                        pontos += 100
                        break

            nave.desenhar()

            for tiro in tiros:
                tiro.desenhar()
            for ast in asteroides:
                ast.desenhar()
            texto_pontos = fonte.render(f"Pontos: {pontos}", True, BRANCO)
            texto_vidas = fonte.render(f"Vidas: {vidas}", True, VERMELHO)
            TELA.blit(texto_pontos, (10, 10))
            TELA.blit(texto_vidas, (10, 45))
        else:
            texto_game_over = fonte_grande.render("GAME OVER", True, VERMELHO)
            texto_reiniciar = fonte.render("Pressione R para reiniciar", True, BRANCO)
            texto_pontos = fonte.render(f"Pontuação final: {pontos}", True, BRANCO)
            TELA.blit(texto_game_over, (LARGURA // 2 - 170, ALTURA // 2 - 80))
            TELA.blit(texto_pontos, (LARGURA // 2 - 120, ALTURA // 2))
            TELA.blit(texto_reiniciar, (LARGURA // 2 - 150, ALTURA // 2 + 50))

        pygame.display.flip()

    pygame.quit()

In [ ]:
if __name__ == "__main__":
    main()

KeyboardInterrupt: 

: 